# FoodShock — Cross-Country ASEAN Food Price Forecasting & Early Shock Detection System

**Machine Learning Solution Challenge (MLSC) – Cyan Academic Competition 2026**  
**Organizer:** Enterprise Data Management (EDM) Laboratory  
**Cabang:** Machine Learning Solution Challenge (EDM Lab)  
**Target SDG:** **SDG 2 — Zero Hunger**  
**Solution Name:** FoodShock  
**Focus:** Residual Forecasting + Early Food-Price Shock Detection across ASEAN

---

## 1. Latar Belakang & Keterkaitan SDG 2 — Zero Hunger

Ketahanan pangan di kawasan ASEAN menghadapi tekanan struktural (perubahan iklim, gangguan rantai pasok, volatilitas kurs) dan tekanan siklikal (musiman panen, permintaan festival). Fluktuasi harga pangan pokok — khususnya beras, daging ayam/sapi/babi, dan minyak goreng — secara langsung memengaruhi daya beli rumah tangga miskin, inflasi pangan, dan stabilitas sosial.

Data WFP/HDX 2015–2025 menunjukkan heterogenitas harga yang tinggi antar negara dan antar jenis pasar (Retail vs Wholesale). Shock harga yang tidak terdeteksi dini mengurangi efektivitas intervensi kebijakan (operasi pasar, pelepasan stok, subsidi tepat sasaran).

**Keterkaitan konkret dengan SDG 2:**
- Target 2.1 & 2.2: akses pangan aman, bergizi, dan terjangkau bagi semua.
- Sistem peringatan dini berbasis machine learning pada level panel antar-negara memberikan instrumen operasional bagi ASEAN Food Security Information System (AFSIS) dan kementerian pertanian/perdagangan nasional.


## Shock Threshold Justification (+5% MoM, upward only)

1. **SDG 2.c:** only upward jumps (`target_pct_change_t1 > +0.05`) are shocks.
2. **Empirical tail:** P90 ≈ +4.44%, P95 ≈ +7.20%; +5% is FoodShock’s operational tail rule (inspired by ALPS/AFSIS language; not a full ALPS re-implementation).
3. **Thresholds (V5):** F1-optimal mean ≈ **0.40**; **operational** mean ≈ **0.63** (max recall s.t. precision ≥ 0.20 on month-block validation; else $F_{0.5}$). **5/6** folds used the precision floor; **1/6** used $F_{0.5}$ fallback.
4. **FDR vs FPR:** FDR = 1−Precision (false alerts among issued alerts); FPR = FP/(FP+TN) (false alerts among true non-shocks). Operational mode targets lower FPR.


## 2. Audit Cakupan Data ASEAN (2015–2025) & Standardisasi

### Aturan yang diterapkan
- Hanya seri (`countryiso3` + `market` + `commodity_group` + `pricetype` + `priceflag` + `unit`) dengan **≥ 60 bulan** observasi.
- 7 negara ASEAN: IDN, KHM, LAO, PHL, THA, TLS, MMR.
- 5 kelompok komoditas baku yang didukung minimal 3 negara: **Rice, Chicken Meat, Beef Meat, Pork Meat, Cooking Oil**.
- Harga dalam **USD**; unit distandardisasi (KG untuk beras/daging, L untuk minyak; konversi 5 L dan KG minyak dengan densitas ≈ 0.92).
- Pemisahan eksplisit `pricetype` (Retail / Wholesale) dan `priceflag`.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

country_cov = pd.read_csv('../data/fpma_wfp_country_coverage.csv')
com_cov = pd.read_csv('../data/fpma_wfp_commodity_coverage.csv')
print("=== Country Coverage (series ≥ 60 months) ===")
display(country_cov)
print("\n=== Commodity Group Coverage ===")
display(com_cov)


=== Country Coverage (series ≥ 60 months) ===
  countryiso3  n_series  n_commodity_groups  avg_months  max_months    min_date    max_date
0         PHL       305                   4     68.8262         116  2015-01-15  2025-12-15
1         IDN       100                   4     60.0800          62  2015-01-15  2024-05-15
2         LAO        82                   5    111.1707         127  2015-01-15  2025-12-15
3         KHM        69                   3     69.2609          88  2015-01-15  2025-12-15
4         MMR        17                   2     77.3529         105  2015-01-15  2025-12-15
5         TLS         7                   3     71.1429          94  2015-01-15  2025-12-15
6         THA         2                   2     63.0000          63  2015-01-15  2020-03-15

=== Commodity Group Coverage ===
  commodity_group  n_series  n_countries                    countries  avg_months  avg_price_usd
0            Rice       204            7  IDN,KHM,LAO,MMR,PHL,THA,TLS     71.1127      

**Catatan penting mengenai IDN & THA:**

- **IDN** seri berakhir sekitar Mei 2024; **THA** sekitar Maret 2020.
- Pada skema **expanding-window walk-forward** (cutoff 2019–2024), keduanya **tetap masuk evaluasi out-of-sample** pada fold di mana data uji masih tersedia (mis. THA pada fold 2019-06; IDN hingga fold sebelum data habis).
- Fold terakhir (2024-06) hanya mencakup negara dengan observasi setelah Juni 2024 (KHM, LAO, MMR, PHL, TLS) — konsisten dengan ketersediaan data, bukan eksklusi sewenang-wenang.


## 3. Calendar-Safe Feature Engineering (Zero Data Leakage)

### Prinsip
- Re-indexing setiap seri ke kalender bulanan penuh **tanpa forward-fill harga mentah**.
- Semua lag dan rolling statistics dihitung dari `shift(1)` sehingga fitur pada waktu *t* hanya menggunakan informasi hingga *t−1*.
- Target residual: $\Delta y_{t+1} = y_{t+1} - y_t$ (menghilangkan bias level antar komoditas yang memiliki skala harga sangat berbeda).
- Rekonstruksi prediksi akhir: $\hat{y}_{t+1} = y_t + \widehat{\Delta y}_{t+1}$.

### Fitur yang dihasilkan
- Lags: `lag_1`, `lag_2`, `lag_3`, `lag_6`, `lag_12`
- Rolling (dari shift(1)): `rolling_mean_3/6`, `rolling_std_6`, `rolling_min_6`, `rolling_max_6`, `price_to_mean6`, `cv_6`
- Momentum: `pct_change_1m`, `pct_change_3m`, `pct_change_12m`
- Musiman: `month_sin`, `month_cos`
- Konteks: `countryiso3`, `commodity_group`, `pricetype`, `priceflag`


In [2]:
model_ready = pd.read_csv('../data/foodshock_asean_model_ready.csv')
model_ready['date'] = pd.to_datetime(model_ready['date'])
print("Shape:", model_ready.shape)
print("Date range:", model_ready['date'].min().date(), "→", model_ready['date'].max().date())
print("Countries:", sorted(model_ready['countryiso3'].unique()))
print("Commodity groups:", sorted(model_ready['commodity_group'].unique()))
print("\nSample of engineered features:")
display(model_ready[['date','countryiso3','commodity_group','usdprice','lag_1','lag_12',
                     'rolling_mean_6','cv_6','pct_change_1m','month_sin',
                     'target_price_usd_t1']].head(6))


Shape: (27950, 39)
Date range: 2016-02-01 → 2025-11-01
Countries: ['IDN', 'KHM', 'LAO', 'MMR', 'PHL', 'THA', 'TLS']
Commodity groups: ['Beef Meat', 'Chicken Meat', 'Cooking Oil', 'Pork Meat', 'Rice']

Sample of engineered features:


,date,countryiso3,commodity_group,usdprice,lag_1,lag_12,rolling_mean_6,cv_6,pct_change_1m,month_sin,target_price_usd_t1
0,2016-02-01,IDN,Beef Meat,7.8700,7.5000,7.4300,7.3717,0.0242,0.0331,0.8660,8.0200
1,2016-02-01,IDN,Chicken Meat,2.9800,2.9800,2.8000,2.7783,0.0636,0.0876,0.8660,2.7500
2,2016-02-01,IDN,Cooking Oil,1.0000,0.9600,1.0900,0.9700,0.0261,0.0105,0.8660,1.0200
3,2016-02-01,IDN,Rice,1.0000,0.9600,1.0000,0.9417,0.0288,0.0213,0.8660,1.0100
4,2016-02-01,KHM,Rice,0.3500,0.3500,0.3200,0.3717,0.0575,0.0000,0.8660,0.3500
5,2016-02-01,KHM,Rice,0.2700,0.3200,0.3000,0.3333,0.0727,-0.0588,0.8660,0.3000


### Tren Harga Rata-rata per Kelompok Komoditas
![EDA Trends](../figures/eda_trends.png)


## 4. Residual Forecasting & Walk-Forward Evaluation (V4)

- **Target:** residual from current price $y_t$ (`usdprice`): $\Delta y_{t+1}=y_{t+1}-y_t$.
- **Validation:** 6-fold expanding walk-forward; **encoders fit only on train fold**.
- **Quantiles:** P10/P90 trained each fold with **conformal** expansion from month-block calibration → overall OOS PICP ≈ **80.2%**.
- **Honest baseline:** Persistence ($y_t$) remains competitive on MAE; CatBoost leads on RMSE (SS_RMSE ≈ +1%) but **panel DM p ≈ 0.35** (not significant at 5%).


In [3]:
results = pd.read_csv('../data/model_evaluation_results.csv')
print("=== Walk-Forward Model Comparison (Residual Reconstruction) ===")
sort_col = 'MAE_micro' if 'MAE_micro' in results.columns else ('MAE' if 'MAE' in results.columns else results.columns[1])
display(results.sort_values(sort_col))
print("\nColumns:", list(results.columns))


=== Walk-Forward Model Comparison (Residual Reconstruction) ===

Columns: ['model', 'MAE_micro', 'RMSE_micro', 'sMAPE_micro', 'R2_micro', 'MAE_macro', 'Persistence_MAE', 'Persistence_RMSE', 'SS_RMSE', 'DM_stat', 'DM_pvalue']


,model,MAE_micro,RMSE_micro,sMAPE_micro,R2_micro,MAE_macro,Persistence_MAE,Persistence_RMSE,SS_RMSE,DM_stat,DM_pvalue
1,Persistence (y_t),0.0986,0.1950,3.0411,0.9936,NaN,0.0986,0.1950,0.0000,NaN,NaN
0,CatBoost_WF_Residual,0.1015,0.1931,3.1794,0.9937,0.0990,0.0986,0.1950,0.0099,-0.9313,0.3517
5,RandomForest_WF_Residual,0.1065,0.2021,3.2706,0.9931,0.1026,0.0986,0.1950,-0.0363,2.0991,0.0358
2,XGBoost_WF_Residual,0.1076,0.1991,3.3475,0.9933,0.1055,0.0986,0.1950,-0.0209,-0.4644,0.6424
3,LightGBM_WF_Residual,0.1083,0.2008,3.5051,0.9932,0.1048,0.0986,0.1950,-0.0297,0.5354,0.5924
4,HistGradientBoosting_WF_Residual,0.1088,0.2021,3.4318,0.9931,0.1048,0.0986,0.1950,-0.0362,0.8239,0.4100


### Ablation & Econometric Honesty

| Quantity | Persistence | Best residual (CatBoost) |
|----------|-------------|---------------------------|
| MAE_micro | **0.0986** | 0.1015 |
| RMSE_micro | 0.1950 | **0.1931** (SS_RMSE ≈ +1.0%) |
| Panel DM p-value | — | ≈ **0.35** (n.s. at 5%) |

Near-martingale prices imply limited mean-error gains. FoodShock’s operational layer is the **calibrated interval** (PICP ≈ 80%) and **Stage-2 ranking** (ROC-AUC ≈ 0.63 vs momentum ≈ 0.49).


### Visualisasi Perbandingan Model & Actual vs Predicted
![Model Comparison](../figures/model_comparison.png)
![Actual vs Predicted](../figures/actual_vs_predicted.png)


## 5. Evaluasi Terpisah per Komoditas, Negara, dan Jenis Pasar


In [4]:
print("=== By Commodity Group ===")
display(pd.read_csv('../data/model_eval_by_commodity.csv'))
print("\n=== By Country ===")
display(pd.read_csv('../data/model_eval_by_country.csv'))
print("\n=== By Price Type (Retail vs Wholesale) ===")
display(pd.read_csv('../data/model_eval_by_pricetype.csv'))


=== By Commodity Group ===

=== By Country ===

=== By Price Type (Retail vs Wholesale) ===


,commodity_group,MAE,RMSE,sMAPE,R2,n
0,Beef Meat,0.1579,0.2513,2.4372,0.9529,4994.0000
1,Chicken Meat,0.1273,0.2267,3.8371,0.8571,4186.0000
2,Cooking Oil,0.0662,0.1959,4.3906,0.8036,1709.0000
3,Pork Meat,0.1582,0.2361,3.1659,0.9304,5069.0000
4,Rice,0.0237,0.0391,3.0472,0.9496,7933.0000


,countryiso3,MAE,RMSE,sMAPE,R2,n
0,IDN,0.0679,0.1420,3.0576,0.9975,3780.0000
1,KHM,0.0788,0.1520,4.0675,0.9920,2629.0000
2,LAO,0.1249,0.3063,4.0390,0.9757,3185.0000
3,MMR,0.1491,0.7169,8.4420,0.5884,84.0000
4,PHL,0.1089,0.1688,2.7345,0.9946,14013.0000
5,THA,0.0299,0.0440,3.2249,0.9947,18.0000
6,TLS,0.1338,0.2620,9.6547,0.9444,182.0000


,pricetype,MAE,RMSE,sMAPE,R2,n
0,Retail,0.1031,0.1955,3.1157,0.9936,22506.0000
1,Wholesale,0.0759,0.1497,4.2137,0.9915,1385.0000


**Interpretasi (selaras output Cell 13 — evaluasi walk-forward OOS):**

- **Per jenis pasar:** Retail mendominasi sampel uji (*n* = 22.506) dengan MAE ≈ 0,101; Wholesale lebih kecil (*n* = 1.385) namun MAE lebih rendah (≈ 0,076).
- **Per komoditas:** **Rice** dan **Cooking Oil** memiliki error absolut terendah (harga relatif stabil); **Pork Meat** dan **Beef Meat** lebih volatil sehingga MAE lebih tinggi.
- **Per negara (7 ASEAN, seluruhnya OOS pada fold yang relevan):** PHL mendominasi jumlah observasi uji; **IDN** (*n* ≈ 3.780) dan **THA** (*n* ≈ 18) ikut dievaluasi *out-of-sample* pada fold di mana data uji tersedia — bukan hanya tersimpan di training set. MMR/TLS memiliki sampel tipis sehingga metrik lebih bervariasi.
- Secara keseluruhan, residual walk-forward menjaga akurasi lintas segmen tanpa bias skala antar komoditas.


## 6. Interpretabilitas SHAP Values & Deteksi Dini Food Price Shock

### SHAP Summary
![SHAP Summary](../figures/shap_summary.png)

**Key Insights SHAP:**
- Setelah residual modeling, `lag_1` tetap relevan namun kontribusi fitur **momentum** (`pct_change_*`), **volatilitas** (`rolling_std_6`, `cv_6`), dan **musiman** (`month_sin` / `month_cos`) menjadi lebih menonjol.
- Encoding negara menangkap heterogenitas struktural antar pasar ASEAN.


### Stage-2 Metrics (V5, upward shock)

| Method | Precision | Recall | F1 | ROC-AUC | PR-AUC | FDR | FPR | Thr |
|--------|-----------|--------|-----|---------|--------|-----|-----|-----|
| F1-optimal | 0.121 | 0.641 | 0.204 | 0.627 | 0.135 | 0.879 | 0.457 | 0.40 |
| **Operational** | **0.163** | 0.192 | 0.176 | **0.627** | **0.135** | **0.837** | **0.097** | **0.63** |
| Momentum | 0.136 | 0.133 | 0.135 | 0.494 | 0.102 | 0.864 | 0.083 | 0.05 |

Conformal PICP (all folds) ≈ **80.2%**. App uses **operational** threshold from `data/app_shock_threshold.txt`.


In [5]:
shock = pd.read_csv('../data/model_eval_shock_classification.csv')
print("=== Shock Classification Metrics ===")
display(shock)


=== Shock Classification Metrics ===


,method,Mean_Train_Threshold,Precision,Recall,F1,ROC_AUC,PR_AUC,FDR,FPR,Base_Rate,Support_Shock,Support_Total,Overall_PICP_P10_P90,Mean_Interval_Width,Shock_Definition
0,Stage2_F1_Optimal,0.4000,0.1214,0.6411,0.2041,0.6270,0.1348,0.8786,0.4566,0.0896,2140,23891,0.8016,0.2919,target_pct_change_t1 > +0.05
1,Stage2_Operational_LowFAR,0.6333,0.1626,0.1916,0.1759,0.6270,0.1348,0.8374,0.0971,0.0896,2140,23891,0.8016,0.2919,target_pct_change_t1 > +0.05
2,Baseline_Momentum_pct_change_1m,0.0500,0.1362,0.1332,0.1347,0.4944,0.1019,0.8638,0.0831,0.0896,2140,23891,0.8016,0.2919,target_pct_change_t1 > +0.05


![Shock Residual Distribution](../figures/shock_residual_dist.png)

### Limitasi Empiris & Saran Pengembangan
- Harga komoditas bulanan mendekati *random walk*; Persistence tetap kompetitif pada MAE.
- Coverage tidak merata (THA, MMR relatif tipis) membatasi stabilitas metrik makro.
- Belum tersedia kovariat cuaca/energi yang harmonis di ketujuh negara.
- Precision classifier masih moderat (trade-off recall untuk early warning).

**Arah lanjutan:** ensemble kuantil / conformal prediction, fitur eksternal lintas negara, dan threshold cost-sensitive untuk dashboard AFSIS operasional.


## 7. Conclusions (V5)

1. **CatBoost residual** is the only point model with RMSE below Persistence (0.1931 vs 0.1950; SS_RMSE ≈ +1%); MAE is slightly higher and **panel DM p ≈ 0.35** (not significant at 5%).
2. **Conformal P10–P90** achieves overall OOS PICP ≈ **80.2%**.
3. **Stage-2** outranks momentum on ROC/PR-AUC; operational threshold (mean ≈ **0.63**) lowers **FPR** to ≈ **9.7%** versus F1-mode ≈ 46%.
4. **Users:** AFSIS → national food agencies → stock/FX verification → proportional market action (SDG 2.c).

*FoodShock · MLSC 2026 · SDG 2 / Target 2.c*
